In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("llabhishekll/fraud-email-dataset")

print("Path to dataset files:", path)


Path to dataset files: /Users/ashokpaudelapril/.cache/kagglehub/datasets/llabhishekll/fraud-email-dataset/versions/1


In [2]:
import os

In [3]:
os.listdir(path)

['fraud_email_.csv']

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [5]:
df = pd.read_csv(os.path.join(path, os.listdir(path)[0]))

In [6]:
import shutil

source_path = os.path.join(path, os.listdir(path)[0])
destination_path = os.getcwd()

shutil.copy(source_path, destination_path)


'/Users/ashokpaudelapril/Machine Learning Projects/Email Fraud Detection/fraud_email_.csv'

In [7]:
df.head(10)

,Text,Class
0,Supply Quality China's EXCLUSIVE dimensions at...,1
1,over. SidLet me know. Thx.,0
2,"Dear Friend,Greetings to you.I wish to accost ...",1
3,MR. CHEUNG PUIHANG SENG BANK LTD.DES VOEUX RD....,1
4,Not a surprising assessment from Embassy.,0
5,Monica -Huma Abedin <Huma@clintonemail.com>Tue...,0
6,Pis print.H <hrod17@clintonemail.com>Thursday ...,0
7,Dear Tom--H <hrod17@clintonemail.com>Friday De...,0
8,Greetings from barrister Robert Williams=2CDea...,1
9,FYI. Thanks again for signing the book ---- an...,0


In [8]:
df["Class"].value_counts()

Class
0    6742
1    5187
Name: count, dtype: int64

In [9]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer



In [10]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)       # remove HTML tags
    text = re.sub(r"http\S+|www.\S+", " ", text)  # remove URLs
    text = re.sub(r"[^a-z\s]", "", text)       # remove punctuation/numbers
    text = re.sub(r"\s+", " ", text).strip()   # remove extra spaces
    tokens = text.split()
    tokens = [word for word in tokens if word not in stopwords.words("english")]
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]
    return " ".join(tokens)

In [11]:
df['Text'] = df['Text'].apply(clean_text)

In [12]:
df.head()

,Text,Class
0,suppli qualiti china exclus dimens unbeat pric...,1
1,sidlet know thx,0
2,dear friendgreet youi wish accost request woul...,1
3,mr cheung puihang seng bank ltdde voeux rd bra...,1
4,surpris assess embassi,0


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['Text'])  # texts is a list of preprocessed strings
Y = df['Class']


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.33, random_state=42)
model = LogisticRegression()

In [15]:
model.fit(X_train, y_train)

LogisticRegression()

In [16]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.93      1.00      0.97      2209
           1       1.00      0.91      0.95      1728

    accuracy                           0.96      3937
   macro avg       0.97      0.95      0.96      3937
weighted avg       0.96      0.96      0.96      3937



In [17]:
def spam_detection(text):
    text = clean_text(text)
    text = vectorizer.transform([text])
    result = model.predict(text)
    return result

In [18]:
spam_detection("Dear Friend,Greetings to you.I wish to accost you with a request that would be of immense benefit to both of us. Being an executor of wills, it is possible that we may be tempted to make fortune out of our client's situations, when we cannot help it, or left with no better option. The issue I am presenting to you is a case of my client who willed a fortune to his next-of-kin. It was most unfortunate that he and his next-of-kin died on the same day the 31st October 1999 in an Egyptian airline 990 with other passengers on board. You can confirm this from the website below which was published by BBC WORLD NEWS.WEBSITE.http:/ews.bbc.co.uk/1/hi/world/americas/502503.stm.I am now faced with confusion of who to pass the fortune to.According to the English law, the fortune is supposed to be bequeathed to the government,if nobody comes forward as the next of kin within seven years of the demise of the benefactor of the will. My purpose of contacting you is to seek your acting as the beneficiary of the will, and lay claim to the legacy of $7million, which my deceased client  bequeathed to his next-of-kin. For now, I alone know about his will, as my client had great confidence in me.Everything will be left between you and I. The share would be 25% for you and 75% for me. I would want to give a huge part of my share to the tsunami victims,as this is my primary objective. All I have to do is amend the will or add a codicil to make you the beneficiary to the $7million legacy.Again, I feel that you may apprehensive and consider this amount too big for you to defend. It does not matter, as there are documents to back it up.This is a legacy being passed on to a next-of-kin. As I am not very sure of getting your consent on the issue I prefer not to divulge my full identity so as not to risk being disbarred. The English Bar considers it a breach of the oath of the English Bar. I need not emphasize to you that the sensitivity of this issue need not be toyed with by neglecting its confidentiality. I therefore appeal to you not discuss this request with anybody, even if you decline my request.I look forward to hearing from you soon.Yours trulyStephen Ayling.")

array([1])

In [19]:
import joblib

joblib.dump(vectorizer, "vectorizer.pkl")



['vectorizer.pkl']

In [20]:
joblib.dump(model, "spam_model.pkl")

['spam_model.pkl']